# SAR-to-EO Image Translation — GalaxEye Assignment
**Model:** Pix2Pix U-Net + PatchGAN

---
## ⚡ Every Session Checklist
1. Settings → Accelerator → **GPU T4**
2. Add dataset: `sentinel12-image-pairs-segregated-by-terrain`
3. *(From Session 2 onwards)* Add your **`sar2eo-checkpoints`** dataset as input
4. Run **Cells 1 → 2 → 3 → 4 → 5**
5. Run the training cell(s) for this session
6. After each config finishes, run **CELL SAVE**

## 📅 Session Plan
| Session | Cells | GPU Hours |
|---|---|---|
| Session 1 | INIT + 1-5 + Smoke + 7a + SAVE + 7b + SAVE | ~10 hrs |
| Session 2 | 1-5 + 7c + SAVE + 7d + SAVE | ~12 hrs |
| Session 3 | 1-5 + 7d (resume if needed) + 8 (eval) | ~2 hrs |


In [ ]:
# ================================================================
# CELL INIT — Run ONCE on your very first session only
# Creates the 'sar2eo-checkpoints' dataset on your Kaggle account.
# After this, add that dataset as input in all future sessions.
# ================================================================
import os, json, subprocess

USERNAME = os.environ.get('KAGGLE_USERNAME', '')
assert USERNAME, 'KAGGLE_USERNAME not set — are you in a Kaggle notebook?'

init_dir = '/kaggle/working/ckpt_init'
os.makedirs(init_dir, exist_ok=True)

meta = {
    'id': f'{USERNAME}/sar2eo-checkpoints',
    'licenses': [{'name': 'CC0-1.0'}],
    'title': 'SAR2EO Training Checkpoints'
}
with open(f'{init_dir}/dataset-metadata.json', 'w') as f:
    json.dump(meta, f)
with open(f'{init_dir}/README.txt', 'w') as f:
    f.write('SAR2EO checkpoint storage. Auto-updated during training.')

result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', init_dir, '-u'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
print(f'\n✅ Dataset created: {USERNAME}/sar2eo-checkpoints')
print('Now add this dataset as Input in all future sessions.')

In [ ]:
# ================================================================
# CELL 1: Environment check
# ================================================================
import subprocess, torch, os
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ================================================================
# CELL 2: Install packages
# ================================================================
!pip install lpips pytorch-fid scikit-image pyyaml -q
print('Packages ready.')

In [ ]:
# ================================================================
# CELL 3: Clone repo
# ================================================================
import os, shutil, subprocess

REPO_DIR = '/kaggle/working/sar2eo'
REPO_URL = 'https://github.com/Trafalgar-2006/sar2eo.git'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('Repo cloned. Working dir:', os.getcwd())

In [ ]:
# ================================================================
# CELL 4: Find dataset + restore checkpoints from Kaggle dataset
#
# Auto-restores any checkpoints saved in previous sessions.
# If sar2eo-checkpoints dataset is mounted, checkpoints are pulled in.
# ================================================================
import os, shutil, glob
from pathlib import Path

# --- Locate Sentinel dataset ---
DATASET_ROOT = None
for base in sorted(Path('/kaggle/input').iterdir()):
    candidates = [base] + [d for d in base.iterdir() if d.is_dir()] if base.is_dir() else []
    for candidate in candidates:
        if Path(candidate).is_dir():
            subdirs = {d.name.lower() for d in Path(candidate).iterdir() if d.is_dir()}
            if subdirs & {'agri', 'urban', 'grassland', 'barrenland'}:
                DATASET_ROOT = str(candidate)
                break
    if DATASET_ROOT:
        break

assert DATASET_ROOT, (
    'Sentinel dataset not found!\n'
    'Add: requiemonk/sentinel12-image-pairs-segregated-by-terrain'
)
print(f'✅ Dataset : {DATASET_ROOT}')

# --- Restore checkpoints from sar2eo-checkpoints dataset ---
restored = 0
for pth in sorted(glob.glob('/kaggle/input/sar2eo-checkpoints/**/*.pth', recursive=True)):
    parts = Path(pth).parts
    # find ablation folder name (l1_only / l1_adv / l1_adv_fft / full)
    ablation = None
    for p in parts:
        if p in ('l1_only', 'l1_adv', 'l1_adv_fft', 'full'):
            ablation = p
            break
    if ablation is None:
        continue
    dest_dir = f'/kaggle/working/sar2eo/checkpoints/{ablation}'
    os.makedirs(dest_dir, exist_ok=True)
    dest = f'{dest_dir}/{Path(pth).name}'
    if not os.path.exists(dest):
        shutil.copy2(pth, dest)
        sz = os.path.getsize(dest) / 1e6
        print(f'  Restored: {ablation}/{Path(pth).name} ({sz:.0f} MB)')
        restored += 1

if restored == 0:
    print('No previous checkpoints found — starting fresh.')
else:
    print(f'\n✅ Restored {restored} checkpoint file(s).')

print()
print('Current checkpoints:')
for f in sorted(glob.glob('/kaggle/working/sar2eo/checkpoints/**/*.pth', recursive=True)):
    sz = os.path.getsize(f) / 1e6
    print(f'  {f.replace("/kaggle/working/sar2eo/checkpoints/","")} ({sz:.0f} MB)')
if not list(glob.glob('/kaggle/working/sar2eo/checkpoints/**/*.pth', recursive=True)):
    print('  (none)')

In [ ]:
# ================================================================
# CELL 5: Configure paths
# ================================================================
import yaml

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['dataset_type']    = 'kaggle'
cfg['data']['kaggle_root']     = DATASET_ROOT
cfg['data']['train_terrain']   = ['agri', 'barrenland', 'grassland']
cfg['data']['val_terrain']     = ['urban']
cfg['data']['test_terrain']    = ['urban']
cfg['data']['subset_size']     = None
cfg['data']['num_workers']     = 2
cfg['training']['batch_size']  = 4
cfg['training']['epochs']      = 100
cfg['paths']['checkpoint_dir'] = '/kaggle/working/sar2eo/checkpoints'
cfg['paths']['output_dir']     = '/kaggle/working/sar2eo/outputs'

with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('✅ Config ready')
print(f'  Dataset     : {DATASET_ROOT}')
print(f'  Checkpoints : /kaggle/working/sar2eo/checkpoints')
print(f'  Batch size  : {cfg["training"]["batch_size"]}')
print(f'  Epochs      : {cfg["training"]["epochs"]}')

In [ ]:
# ================================================================
# CELL 6: Smoke test — Run ONCE on Session 1 only
# Skip this in Session 2 onwards.
# ================================================================
import yaml, subprocess, os

with open('config.yaml') as f:
    cfg_s = yaml.safe_load(f)

cfg_s['data']['subset_size']     = 100
cfg_s['training']['epochs']      = 2
cfg_s['training']['val_freq']    = 1
cfg_s['training']['save_freq']   = 2
cfg_s['active_ablation']         = 'full'
cfg_s['paths']['checkpoint_dir'] = '/kaggle/working/smoke_ckpts'
cfg_s['paths']['output_dir']     = '/kaggle/working/smoke_out'

with open('config_smoke.yaml', 'w') as f:
    yaml.dump(cfg_s, f)

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

print('Running smoke test (2 epochs, 100 pairs)...')
r = subprocess.run(
    ['python', '-u', 'train.py', '--config', 'config_smoke.yaml', '--ablation', 'full'],
    env=env
)
if r.returncode != 0:
    raise RuntimeError('❌ Smoke test FAILED')
print('\n✅ Smoke test PASSED — ready for full training.')

In [ ]:
# ================================================================
# CELL 7a: Config A — L1 only (baseline)
# Estimated: ~3.7 hrs | Auto-resumes if checkpoint exists
# ================================================================
import subprocess, os
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

print('=' * 50)
print('Config A: L1 only (no GAN)')
print('=' * 50)
r = subprocess.run(
    ['python', '-u', 'train.py', '--config', 'config.yaml', '--ablation', 'l1_only'],
    env=env
)
print('\nConfig A', '✅ DONE' if r.returncode == 0 else '❌ FAILED')
print('→ Run CELL SAVE now to push checkpoints.')

In [ ]:
# ================================================================
# CELL 7b: Config B — L1 + Adversarial
# Estimated: ~6.3 hrs | Auto-resumes if checkpoint exists
# ================================================================
import subprocess, os
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

print('=' * 50)
print('Config B: L1 + Adversarial')
print('=' * 50)
r = subprocess.run(
    ['python', '-u', 'train.py', '--config', 'config.yaml', '--ablation', 'l1_adv'],
    env=env
)
print('\nConfig B', '✅ DONE' if r.returncode == 0 else '❌ FAILED')
print('→ Run CELL SAVE now to push checkpoints.')

In [ ]:
# ================================================================
# CELL 7c: Config C — L1 + Adversarial + FFT
# Estimated: ~6.3 hrs | Auto-resumes if checkpoint exists
# ================================================================
import subprocess, os
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

print('=' * 50)
print('Config C: L1 + Adversarial + FFT')
print('=' * 50)
r = subprocess.run(
    ['python', '-u', 'train.py', '--config', 'config.yaml', '--ablation', 'l1_adv_fft'],
    env=env
)
print('\nConfig C', '✅ DONE' if r.returncode == 0 else '❌ FAILED')
print('→ Run CELL SAVE now to push checkpoints.')

In [ ]:
# ================================================================
# CELL 7d: Config D — Full model (MAIN SUBMISSION)
# Estimated: ~6.5 hrs | Auto-resumes if checkpoint exists
# ================================================================
import subprocess, os
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

print('=' * 50)
print('Config D: Full model (L1 + Adv + FFT + VGG)')
print('=' * 50)
r = subprocess.run(
    ['python', '-u', 'train.py', '--config', 'config.yaml', '--ablation', 'full'],
    env=env
)
print('\nConfig D', '✅ DONE' if r.returncode == 0 else '❌ FAILED')
print('→ Run CELL SAVE now to push checkpoints.')

In [ ]:
# ================================================================
# CELL SAVE — Run after EACH config finishes
#
# Pushes all checkpoints to your 'sar2eo-checkpoints' Kaggle dataset.
# They are permanently saved and available in the next session.
# ================================================================
import os, json, shutil, glob, subprocess

USERNAME = os.environ.get('KAGGLE_USERNAME', '')
assert USERNAME, 'KAGGLE_USERNAME not set'

DATASET_ID = f'{USERNAME}/sar2eo-checkpoints'
CKPT_SRC   = '/kaggle/working/sar2eo/checkpoints'
UPLOAD_DIR = '/kaggle/working/ckpt_push'

# Set up upload directory
if os.path.exists(UPLOAD_DIR):
    shutil.rmtree(UPLOAD_DIR)
os.makedirs(UPLOAD_DIR)

# Dataset metadata
meta = {
    'id': DATASET_ID,
    'licenses': [{'name': 'CC0-1.0'}],
    'title': 'SAR2EO Training Checkpoints'
}
with open(f'{UPLOAD_DIR}/dataset-metadata.json', 'w') as f:
    json.dump(meta, f)

# Copy all checkpoints
if os.path.exists(CKPT_SRC):
    shutil.copytree(CKPT_SRC, f'{UPLOAD_DIR}/checkpoints')

# List what will be pushed
pth_files = sorted(glob.glob(f'{UPLOAD_DIR}/checkpoints/**/*.pth', recursive=True))
print(f'Pushing {len(pth_files)} checkpoint file(s) to {DATASET_ID}:')
for f in pth_files:
    sz = os.path.getsize(f) / 1e6
    print(f'  {f.replace(UPLOAD_DIR+"/checkpoints/", "")} ({sz:.0f} MB)')

if not pth_files:
    print('No .pth files found — nothing to push.')
else:
    # Push new version to Kaggle dataset
    result = subprocess.run(
        ['kaggle', 'datasets', 'version',
         '-p', UPLOAD_DIR,
         '-m', f'{len(pth_files)} checkpoints updated'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f'\n✅ All checkpoints pushed to {DATASET_ID}')
        print('They are permanently saved and will be available in the next session.')
    else:
        print(f'❌ Push failed:\n{result.stderr}')
        print('Tip: Make sure you ran CELL INIT on your first session.')

In [ ]:
# ================================================================
# CELL 8: Evaluate all configs (run after all training is done)
# ================================================================
import subprocess, os
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

for ablation in ['l1_only', 'l1_adv', 'l1_adv_fft', 'full']:
    ckpt = f'/kaggle/working/sar2eo/checkpoints/{ablation}/best.pth'
    if not os.path.exists(ckpt):
        print(f'  Skip {ablation} — no best.pth')
        continue
    print(f'\n{"="*45}')
    print(f'Evaluating: {ablation}')
    print(f'{"="*45}')
    subprocess.run([
        'python', '-u', 'eval.py',
        '--config', 'config.yaml',
        '--weights', ckpt,
        '--split', 'test'
    ], env=env)

print('\n✅ All evaluations complete.')